# Real saved checkpoint reload integration

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from evidence_verification import verify_checkpoint,REFERENCE
for run,attempt in [('P1-TRAIN-SMOKE','attempt-02'),('P1-TASK','attempt-01'),('P1-MAX','attempt-01')]:print(json.dumps(verify_checkpoint(REFERENCE/'runs'/run/attempt),indent=2))
new=TRACE_ROOT/'outputs/runs/checkpoint_check/attempt-01'
if (new/'COMPLETE.json').exists():print('NEW RUN:',json.dumps(verify_checkpoint(new),indent=2))
print('Real saved checkpoint reload integration definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Independent migration verification definitions/execution completed.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Measured PPO training and checkpoint evaluation definitions/execution completed.


{
  "checkpoint": "outputs/reference/runs/P1-TRAIN-SMOKE/attempt-02/train/checkpoint_4096.zip",
  "decisions": 4096,
  "rollouts": 2,
  "actions_reloaded": 39,
  "sha256": "464340a1ef13ca9a30e4a8d4a584647ca3f0517f8dc10a265654ce6a1846e710"
}
{
  "checkpoint": "outputs/reference/runs/P1-TASK/attempt-01/train/checkpoint_51200.zip",
  "decisions": 51200,
  "rollouts": 25,
  "actions_reloaded": 486,
  "sha256": "6dfc8c9319fb15d65dfa9f76d29119f1f3240d72e5b82b349b5c31c2b9427e97"
}
{
  "checkpoint": "outputs/reference/runs/P1-MAX/attempt-01/train/checkpoint_51200.zip",
  "decisions": 51200,
  "rollouts": 25,
  "actions_reloaded": 486,
  "sha256": "c3206e4862338ed769fe69125844c1f3a78689a19f3cff084994994cb49df11f"
}
NEW RUN: {
  "checkpoint": "outputs/runs/checkpoint_check/attempt-01/train/checkpoint_4096.zip",
  "decisions": 4096,
  "rollouts": 2,
  "actions_reloaded": 39,
  "sha256": "3696cc289c051e774839faed9d00580a30d4628c40616c424198bb4752bda75a"
}
Real saved checkpoint reload integration d